In [1]:
import os
os.chdir("application")

In [2]:
from langchain_milvus_ingest import store_chunks_to_milvus

C:\Users\USER\Miniconda3\envs\kpn-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from ingest.chunking import pages_to_chunks
from ingest.pdf_with_tables import extract_pdf_content

In [4]:
from pymilvus import connections, utility, Collection

In [5]:
from config import embedding_model, milvus_host, milvus_port, collection_name

In [6]:
folder_path = '../assets'

In [ ]:
# for filename in os.listdir(folder_path):
#     filepath = os.path.join(folder_path, filename)
#     print(filepath)
#     pdf_pages = extract_pdf_content(filepath)
#     filename = os.path.basename(filepath)
#     all_chunks = pages_to_chunks(pdf_pages, source=filename, language='id')
    

In [7]:
all_chunks = []
for filename in os.listdir(folder_path):
    filepath = os.path.join(folder_path, filename)
    if filepath.endswith('.pdf'):
        print(filepath)
        pdf_pages = extract_pdf_content(filepath)

        filename = os.path.basename(filepath)
        chunks = pages_to_chunks(pdf_pages, source=filename)
        all_chunks.extend(chunks)

../assets\CP-Bantuan-Pengobatan-2025.pdf
../assets\CP-Cuti-dan-Izin-Khusus.pdf
../assets\CP_Perjalanan Dinas.pdf


In [ ]:
all_chunks[-50]

In [ ]:
for chunk in all_chunks:
#     if chunk['source']=='CP_Perjalanan Dinas.pdf' and chunk['element_type']=='table_row' and chunk['page']==25:
    if chunk['source']=='CP_Perjalanan Dinas.pdf' and chunk['page']==26:
        print(chunk['chunk_text'], '\n\n')

In [ ]:
for chunk in all_chunks:
    if chunk['source']=='CP-Bantuan-Pengobatan-2025.pdf':
        print(chunk['chunk_text'], '\n\n')

In [8]:
HOST = milvus_host
PORT = milvus_port
COL = collection_name  # your collection name

# 1) connect
connections.connect(alias="default", host=HOST, port=PORT)

# 2) show collections
print("Collections:", utility.list_collections())

# 3) does collection exist?
print("Has collection:", utility.has_collection(COL))

# 4) inspect schema (if exists)
if utility.has_collection(COL):
    c = Collection(COL)
    print("Num entities:", c.num_entities)
    print("Schema fields:")
    for f in c.schema.fields:
        print("  ", f.name, f.dtype, getattr(f, "dim", None))

# 5) run a checksum query (example)
if utility.has_collection(COL):
    rows = c.query(expr='checksum in ["sha256-some"]', output_fields=["checksum","metadata"])
    print(rows)
coll = c

Collections: ['pdf_docs_test', 'milvus_langchain_test', 'company_policies']
Has collection: True
Num entities: 154
Schema fields:
   id 21 None
   embedding 101 768
   checksum 21 None
   text 21 None
   metadata 23 None
data: [], extra_info: {}


In [14]:
ls_col = utility.list_collections()
ls_col

[]

In [13]:
# for col in ls_col:
#     utility.drop_collection(col)

In [9]:
if utility.has_collection("company_policies"):
    utility.drop_collection("company_policies")
    print("deleted")

deleted


In [15]:
skipped, inserted = store_chunks_to_milvus(
    chunks=all_chunks,
    collection_name=collection_name,
    milvus_host=milvus_host,
    milvus_port=milvus_port,
    embedding_model=embedding_model,
    embed_batch_size=32,
    dedupe_with_milvus=True,
    dry_run=False
)
print("skipped, inserted =", skipped, inserted)


INFO:langchain_milvus_ingest:Created Milvus collection 'company_policies' with dim=3072
INFO:langchain_milvus_ingest:Finished ingest: inserted=154, skipped=0


skipped, inserted = 0 154


In [ ]:
stop

In [17]:
coll.schema

{'auto_id': False, 'description': 'RAG KB - chunks with Gemini embeddings', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 128}, 'is_primary': True, 'auto_id': False}, {'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}, {'name': 'checksum', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 128}}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}, {'name': 'metadata', 'description': '', 'type': <DataType.JSON: 23>}], 'enable_dynamic_field': False}

In [21]:
coll.schema.fields

[{'name': 'id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 128}, 'is_primary': True, 'auto_id': False},
 {'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}},
 {'name': 'checksum', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 128}},
 {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}},
 {'name': 'metadata', 'description': '', 'type': <DataType.JSON: 23>}]

In [ ]:
# assume coll exists from previous snippet
checksum = "sha256-your-checksum-here"
rows = coll.query(expr=f'checksum in ["{checksum}"]', output_fields=["id","checksum","metadata"])
print("Query by checksum ->", rows)
# or by chunk_id if you stored it as primary id:
chunk_id = "c-12345"
rows2 = coll.query(expr=f'id in ["{chunk_id}"]', output_fields=["id","checksum","metadata"])
print("Query by id ->", rows2)


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings  # or fallback import

In [ ]:
api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
# many wrappers read env automatically; if yours needs explicit key, pass google_api_key=api_key
emb = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=api_key)

In [ ]:
query_text = "Berapa biaya akomodasi untuk perjalanan dinas golongan 6 ke Sumatera?"
qvec = emb.embed_documents([query_text])[0]   # a single vector (list of floats)
print("Query vector dim:", len(qvec))

In [ ]:
# coll from earlier
TOP_K = 5
# search params for HNSW/COSINE index — tune ef (higher -> higher recall, slower)
search_params = {"metric_type": "COSINE", "params": {"ef": 100}}

results = coll.search(
    data=[qvec],                 # list of query vectors (can search multiple at once)
    anns_field="embedding",      # the name of your vector field
    param=search_params,
    limit=TOP_K,
    output_fields=["id", "checksum", "metadata"]
)

# results is a list (one item per query) of "hits" lists
for hits in results:
    for hit in hits:
        # printing Hit attributes - depending on pymilvus version use .id or .entity.get("id")
        try:
            entity_id = hit.id
        except Exception:
            entity_id = hit.entity.get("id") if hasattr(hit, "entity") else None
        # score/distance: with COSINE higher is more similar (closer to 1). Many APIs expose .distance or .score
        score = getattr(hit, "score", None) or getattr(hit, "distance", None)
        metadata = hit.entity.get("metadata") if hasattr(hit, "entity") else None
        print("id:", entity_id, "score:", score, "checksum:", hit.entity.get("checksum") if hasattr(hit,"entity") else None, "meta:", metadata)


In [ ]:
for hits in results:
    for hit in hits:
        # printing Hit attributes - depending on pymilvus version use .id or .entity.get("id")
        try:
            entity_id = hit.id
        except Exception:
            entity_id = hit.entity.get("id") if hasattr(hit, "entity") else None
        # score/distance: with COSINE higher is more similar (closer to 1). Many APIs expose .distance or .score
        score = getattr(hit, "score", None) or getattr(hit, "distance", None)
        metadata = hit.entity.get("metadata") if hasattr(hit, "entity") else None
#         print("id:", entity_id, "score:", score, "checksum:", hit.entity.get("checksum") if hasattr(hit,"entity") else None, "meta:", metadata)
        print(hit.get("metadata")['chunk_text'], '\n')


In [ ]:
# Query all chunks from page 3
rows = coll.query(
    expr="page == 3", 
    output_fields=["id", "page", "metadata"]
)
print("Results:", rows)

In [ ]:
all_chunks[-10]